In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle


def ReadFile(name):
    d = np.loadtxt(name)
    return d[0, 1:], d[1:, 0], d[1:, 1:]     # X, Y, M[j, i]

def ReadMaterial(name):
    L = open(name).readlines().splitlines()
    nx,ny,z = [int(x) for x in L[0].split()]
    
    return d[0, 1:], d[1:, 0], d[1:, 1:]     # X, Y, M[j, i]

X, Y, Phi = ReadFile('PhiMatrix.dat')
_, _, dx  = ReadFile('dxPhiMatrix.dat')
_, _, dy  = ReadFile('dyPhiMatrix.dat')

Ex, Ey = -dx, -dy                             # E = -grad(Phi)
X, Y = 1000*X, 1000*Y                         # convert to mm
# Deep-metal nodes are exactly 0 in both components: hide them
mask = (dx == 0) & (dy == 0)
Phi = np.ma.array(Phi, mask=mask) 
Ex = np.ma.array(Ex, mask=mask)
Ey = np.ma.array(Ey, mask=mask)


In [96]:
p = np.pad(mask,1,constant_values=False)
near = np.zeros_like(mask, dtype=bool)
near = (p[2:, 1:-1] | p[:-2, 1:-1] | p[1:-1, 2:] | p[1:-1, :-2] |    # up, down, right, left
        p[2:, 2:]   | p[2:, :-2]   | p[:-2, 2:]  | p[:-2, :-2]) & ~mask
Phi_ring = np.ma.array(Phi, mask=~near)
dx_ring = np.ma.array(dx, mask=~near)
dy_ring = np.ma.array(dy, mask=~near)


XX, YY = np.meshgrid(X, Y)

origin = np.argmin(XX**2 + YY**2,axis=None)  # find the node closest to the origin
theta = np.arctan2(YY, XX)  # angle of the origin node

Dcentral_Phi = np.cos(theta)*dx + np.sin(theta)*dy

Dcentral_Phi_borders = np.ma.array(Dcentral_Phi, mask=~near)

s =(np.cos(theta)*dx_ring + np.sin(theta)*dy_ring).sum()


Phi_ring.sum(), dx_ring.sum(), dy_ring.sum(), Dcentral_Phi_borders.sum()


(np.float64(-597.97976465),
 np.float64(-1040.13354257226),
 np.float64(-1384.2108990069203),
 np.float64(880.1687493775706))